In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, precision_recall_curve

# 1. DATA LOADING & PREPROCESSING
df = pd.read_csv('HTRU_2.csv', header=None)
df.columns = ['mean_ip','std_ip','kurt_ip','skew_ip', 'mean_dm','std_dm','kurt_dm','skew_dm','label']

X = df.drop('label', axis=1).values
y = df['label'].values

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)
X_dev, X_test, y_dev, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_dev = scaler.transform(X_dev)
X_test = scaler.transform(X_test)

weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weights = {0: weights[0], 1: weights[1]}

# 2. MODEL BUILDING & TRAINING
model = tf.keras.Sequential([
    tf.keras.layers.Dense(16, activation='relu', kernel_initializer='he_normal', input_shape=(8,)),
    tf.keras.layers.Dense(8, activation='relu', kernel_initializer='he_normal'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), 
              loss='binary_crossentropy', 
              metrics=['accuracy'])

history = model.fit(
    X_train, y_train,
    validation_data=(X_dev, y_dev),
    epochs=30,
    batch_size=32,
    class_weight=class_weights,
    verbose=0
)

# 3. PERFORMANCE ANALYSIS
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('Model Convergence')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.show()

y_dev_proba = model.predict(X_dev, verbose=0).flatten()
precision, recall, thresholds = precision_recall_curve(y_dev, y_dev_proba)

mask = recall[:-1] >= 0.90
best_threshold = thresholds[mask][np.argmax(precision[mask])]
print(f"Optimal Threshold: {best_threshold:.3f}")

# 4. FINAL TESTING
y_test_proba = model.predict(X_test, verbose=0).flatten()
y_test_pred = (y_test_proba >= best_threshold).astype(int)

print("
--- FINAL CLASSIFICATION REPORT ---")
print(classification_report(y_test, y_test_pred, digits=3))

